# ViT Exploration on FineBadminton Frames

This notebook explores what Vision Transformers can contribute to the badminton tactical strategy pipeline, using 5 rallies as a testbed.

## Potential ViT Use Cases in This Pipeline

| # | Use Case | Why Interesting |
|---|----------|-----------------|
| 1 | **Attention map visualisation** (DINO) | See if self-supervised ViT naturally segments players, shuttle, court — without any badminton-specific training |
| 2 | **Zero-shot strategy classification** (CLIP) | Test whether CLIP text prompts can classify raw frames into strategy classes — no labelling required |
| 3 | **CLS-token feature clustering** | Extract ViT CLS-token embeddings for each hit frame; t-SNE/PCA to see if strategy classes cluster from pixels alone |
| 4 | **Shuttle/player saliency** | Which image patches drive ViT predictions? Compare to skeleton locations |
| 5 | **Multi-modal fusion signal** | Are ViT features complementary to ST-GCN skeleton features? Quick correlation check |

Models used:
- `facebook/dino-vitb16` — self-supervised, excellent attention maps
- `openai/clip-vit-base-patch32` — zero-shot text-image matching

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from collections import defaultdict

import torch
import torch.nn.functional as F
from torchvision import transforms

from src.config import FB_FRAMES, FB_ANNOTATIONS, FB_SKELETONS_GDINO_V2

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load 5 Rallies + Annotations

In [ ]:
with open(FB_ANNOTATIONS) as f:
    all_rallies = json.load(f)

FB_STRATEGY_MAP = {
    'intercept': 'intercept',
    'defensive': 'defensive',
    'move to the net': 'move_to_net',
    'to create depth': 'create_depth',
    'passive': 'passive',
}
FB_EXCLUDED = {'deception', 'hesitation', 'seamlessly', 'a high net early shot'}

# Pick first 5 rallies that have annotated shots
RALLY_IDS = [r['video'].replace('.mp4', '') for r in all_rallies[:5]]
print('Selected rallies:', RALLY_IDS)

# Build a flat list of shots with strategy labels
shots = []
for rally in all_rallies:
    vid = rally['video'].replace('.mp4', '')
    if vid not in RALLY_IDS:
        continue
    for hit in rally['hitting']:
        raw_strategy = hit.get('strategy', '')
        strategy = FB_STRATEGY_MAP.get(raw_strategy)
        if strategy is None:
            continue  # excluded or unlabelled
        shots.append({
            'rally': vid,
            'hit_frame': hit['hit_frame'],
            'start_frame': hit['start_frame'],
            'end_frame': hit['end_frame'],
            'strategy': strategy,
            'hit_type': hit.get('hit_type', ''),
            'hitter': hit.get('hitter', ''),
            'ball_area': hit.get('ball_area', ''),
            'comment': hit.get('comment', ''),
        })

print(f'Total labelled shots across 5 rallies: {len(shots)}')
from collections import Counter
print('Strategy distribution:', Counter(s['strategy'] for s in shots))

In [ ]:
def load_frame(rally_id: str, frame_num: int) -> Image.Image:
    path = FB_FRAMES / f'{rally_id}_{frame_num}.jpg'
    if not path.exists():
        raise FileNotFoundError(path)
    return Image.open(path).convert('RGB')

# Quick sanity-check: show one hit frame per strategy
seen = {}
for s in shots:
    if s['strategy'] not in seen:
        seen[s['strategy']] = s

fig, axes = plt.subplots(1, len(seen), figsize=(4*len(seen), 4))
for ax, (strategy, s) in zip(axes, seen.items()):
    img = load_frame(s['rally'], s['hit_frame'])
    ax.imshow(img)
    ax.set_title(f"{strategy}\n{s['hit_type']}", fontsize=9)
    ax.axis('off')
plt.suptitle('One hit frame per strategy class (5 rallies)', y=1.02)
plt.tight_layout()
plt.show()

## 2. DINO ViT — Self-Attention Maps

DINO (self-DIstillation with NO labels) trains ViT to produce meaningful attention maps without supervision. The [CLS] token attention heads tend to highlight semantically meaningful regions — in natural images this often segments foreground objects.

**Question**: Do the DINO attention heads naturally attend to players and the shuttle?

In [ ]:
from transformers import ViTModel, AutoFeatureExtractor

dino_name = 'facebook/dino-vitb16'
dino_extractor = AutoFeatureExtractor.from_pretrained(dino_name)
dino_model = ViTModel.from_pretrained(dino_name, output_attentions=True).to(DEVICE).eval()
print('DINO ViT-B/16 loaded')

In [ ]:
def get_dino_attentions(img: Image.Image):
    """Return last-layer attention maps, shape (heads, h_patches, w_patches)."""
    inputs = dino_extractor(images=img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = dino_model(**inputs)
    # last layer attention: (1, heads, seq_len, seq_len)
    att = out.attentions[-1][0]  # (heads, seq_len, seq_len)
    # CLS token (index 0) attending to all patches
    cls_att = att[:, 0, 1:]  # (heads, num_patches)
    # ViT-B/16 on 224×224 → 14×14 patches
    h = w = int(cls_att.shape[-1] ** 0.5)
    cls_att = cls_att.reshape(-1, h, w).cpu().numpy()
    return cls_att


def plot_dino_attention(img: Image.Image, att_maps: np.ndarray,
                        title: str = '', n_heads: int = 4, ax_img=None):
    """Overlay mean attention map on the image."""
    mean_att = att_maps.mean(axis=0)  # average over heads
    mean_att = (mean_att - mean_att.min()) / (mean_att.max() - mean_att.min() + 1e-8)
    att_resized = np.array(Image.fromarray((mean_att * 255).astype(np.uint8)).resize(
        img.size, Image.BILINEAR)) / 255.0

    if ax_img is None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        ax_img, ax_att = axes
    else:
        fig = ax_img.get_figure()
        ax_att = fig.axes[fig.axes.index(ax_img) + 1]

    ax_img.imshow(img)
    ax_img.set_title('Frame')
    ax_img.axis('off')

    ax_att.imshow(img)
    ax_att.imshow(att_resized, alpha=0.6, cmap='jet')
    ax_att.set_title(f'DINO attention (mean {att_maps.shape[0]} heads)\n{title}')
    ax_att.axis('off')

    return mean_att, att_resized

In [ ]:
# Show DINO attention for one shot per strategy
n_strategies = len(seen)
fig, axes = plt.subplots(n_strategies, 2, figsize=(12, 4*n_strategies))

dino_features = {}  # strategy → CLS token embedding

for row, (strategy, s) in enumerate(seen.items()):
    img = load_frame(s['rally'], s['hit_frame'])
    att = get_dino_attentions(img)

    ax_img = axes[row, 0]
    ax_att = axes[row, 1]

    ax_img.imshow(img)
    ax_img.set_title(f"Strategy: {strategy}\n{s['hit_type']} | {s['hitter']} | {s['ball_area']}", fontsize=9)
    ax_img.axis('off')

    mean_att = att.mean(axis=0)
    mean_att_norm = (mean_att - mean_att.min()) / (mean_att.max() - mean_att.min() + 1e-8)
    att_resized = np.array(Image.fromarray((mean_att_norm * 255).astype(np.uint8)).resize(
        img.size, Image.BILINEAR)) / 255.0
    ax_att.imshow(img)
    ax_att.imshow(att_resized, alpha=0.55, cmap='hot')
    ax_att.set_title('DINO CLS attention (mean over 12 heads)', fontsize=9)
    ax_att.axis('off')

plt.suptitle('DINO ViT-B/16 Attention Maps — Hit Frames by Strategy', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Show individual attention heads for one example (intercept) to see head specialisation
s = seen.get('intercept') or shots[0]
img = load_frame(s['rally'], s['hit_frame'])
att = get_dino_attentions(img)  # (12, 14, 14)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    head_map = att[i]
    head_norm = (head_map - head_map.min()) / (head_map.max() - head_map.min() + 1e-8)
    head_resized = np.array(Image.fromarray((head_norm * 255).astype(np.uint8)).resize(
        img.size, Image.BILINEAR)) / 255.0
    ax.imshow(img)
    ax.imshow(head_resized, alpha=0.6, cmap='hot')
    ax.set_title(f'Head {i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle(f"DINO — All 12 Attention Heads\n{s['strategy']} | {s['hit_type']}", fontsize=12)
plt.tight_layout()
plt.show()

## 3. CLIP Zero-Shot Strategy Classification

CLIP maps images and text into a shared embedding space. We can write descriptive text prompts for each strategy class and see how well they match hit frames — **zero-shot, no training**.

Strategy text prompts are the key design decision here.

In [ ]:
from transformers import CLIPProcessor, CLIPModel

clip_name = 'openai/clip-vit-base-patch32'
clip_processor = CLIPProcessor.from_pretrained(clip_name)
clip_model = CLIPModel.from_pretrained(clip_name).to(DEVICE).eval()
print('CLIP ViT-B/32 loaded')

In [ ]:
# Strategy text prompts — deliberately detailed to test zero-shot quality
STRATEGY_PROMPTS = {
    'intercept': [
        'a badminton player moving forward quickly to intercept the shuttle near the net',
        'badminton player lunging forward to cut off an early shot',
    ],
    'defensive': [
        'a badminton player in a defensive stance at the back of the court',
        'badminton player scrambling to return a smash defensively',
    ],
    'move_to_net': [
        'a badminton player rushing to the net after playing a shot',
        'badminton player approaching the net ready to play a net kill',
    ],
    'create_depth': [
        'a badminton player hitting a high lift or clear to the back of the court',
        'badminton player playing a deep lob to push the opponent to the backcourt',
    ],
    'passive': [
        'a badminton player in a neutral waiting position in the middle of the court',
        'badminton player passively returning without attacking',
    ],
}

# Use mean embedding over multiple prompts per class
def get_text_embeddings(prompt_dict):
    class_embeddings = {}
    for cls, prompts in prompt_dict.items():
        inputs = clip_processor(text=prompts, return_tensors='pt', padding=True).to(DEVICE)
        with torch.no_grad():
            emb = clip_model.get_text_features(**inputs)  # (n_prompts, dim)
        emb = F.normalize(emb, dim=-1).mean(dim=0)
        emb = F.normalize(emb, dim=-1)
        class_embeddings[cls] = emb
    return class_embeddings

text_embs = get_text_embeddings(STRATEGY_PROMPTS)
text_matrix = torch.stack(list(text_embs.values()))  # (5, dim)
class_names = list(text_embs.keys())
print('Text embeddings ready for classes:', class_names)

In [ ]:
def clip_classify(img: Image.Image) -> dict:
    """Return CLIP softmax scores for each strategy class."""
    inputs = clip_processor(images=img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        img_emb = clip_model.get_image_features(**inputs)  # (1, dim)
    img_emb = F.normalize(img_emb, dim=-1)
    sims = (img_emb @ text_matrix.T)[0]  # (5,)
    probs = F.softmax(sims * 100, dim=-1).cpu().numpy()  # temperature=100 like CLIP paper
    return {cls: float(p) for cls, p in zip(class_names, probs)}


# Run CLIP on all shots and compute accuracy
y_true, y_pred, y_probs_all = [], [], []
for s in shots:
    img = load_frame(s['rally'], s['hit_frame'])
    scores = clip_classify(img)
    pred = max(scores, key=scores.get)
    y_true.append(s['strategy'])
    y_pred.append(pred)
    y_probs_all.append(scores)

acc = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
print(f'CLIP zero-shot accuracy: {acc:.1%} ({sum(t==p for t,p in zip(y_true,y_pred))}/{len(y_true)})')

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

all_cls = list(STRATEGY_PROMPTS.keys())
cm = confusion_matrix(y_true, y_pred, labels=all_cls)
disp = ConfusionMatrixDisplay(cm, display_labels=[c.replace('_', '\n') for c in all_cls])
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'CLIP Zero-Shot Confusion Matrix (acc={acc:.1%})')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy breakdown
per_class = defaultdict(lambda: {'correct': 0, 'total': 0})
for t, p in zip(y_true, y_pred):
    per_class[t]['total'] += 1
    if t == p:
        per_class[t]['correct'] += 1

print('Per-class accuracy:')
for cls in all_cls:
    c = per_class[cls]
    print(f"  {cls:20s}: {c['correct']}/{c['total']} = {c['correct']/max(c['total'],1):.1%}")

## 4. CLS Token Embedding Clustering

Extract DINO CLS-token embeddings for all hit frames. If strategy-discriminative information exists in raw pixels, the embeddings should cluster by strategy in t-SNE / PCA space.

Compare to ST-GCN skeleton embeddings as a baseline signal strength check.

In [ ]:
def get_dino_cls(img: Image.Image) -> np.ndarray:
    """Return DINO CLS token embedding, shape (768,)."""
    inputs = dino_extractor(images=img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = dino_model(**inputs)
    return out.last_hidden_state[0, 0].cpu().numpy()  # CLS token


print('Extracting DINO CLS embeddings for all shots...')
embeddings, labels = [], []
for i, s in enumerate(shots):
    img = load_frame(s['rally'], s['hit_frame'])
    emb = get_dino_cls(img)
    embeddings.append(emb)
    labels.append(s['strategy'])
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(shots)}')

embeddings = np.stack(embeddings)  # (N, 768)
print(f'Done. Embedding matrix: {embeddings.shape}')

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

STRATEGY_COLORS = {
    'intercept': '#e41a1c',
    'defensive': '#377eb8',
    'move_to_net': '#4daf4a',
    'create_depth': '#ff7f00',
    'passive': '#984ea3',
}

# PCA first for speed
pca = PCA(n_components=50, random_state=42)
emb_pca = pca.fit_transform(embeddings)

# t-SNE
perplexity = min(30, len(shots) - 1)
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
emb_2d = tsne.fit_transform(emb_pca)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# t-SNE
ax = axes[0]
for cls, col in STRATEGY_COLORS.items():
    idx = [i for i, l in enumerate(labels) if l == cls]
    if idx:
        ax.scatter(emb_2d[idx, 0], emb_2d[idx, 1], c=col, label=cls, s=60, alpha=0.8, edgecolors='k', linewidths=0.4)
ax.set_title('t-SNE of DINO CLS embeddings\n(hit frames, coloured by strategy)', fontsize=11)
ax.legend(fontsize=8)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')

# PCA 2D
ax = axes[1]
for cls, col in STRATEGY_COLORS.items():
    idx = [i for i, l in enumerate(labels) if l == cls]
    if idx:
        ax.scatter(emb_pca[idx, 0], emb_pca[idx, 1], c=col, label=cls, s=60, alpha=0.8, edgecolors='k', linewidths=0.4)
ax.set_title('PCA 2D of DINO CLS embeddings\n(hit frames, coloured by strategy)', fontsize=11)
ax.legend(fontsize=8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')

plt.tight_layout()
plt.show()

In [ ]:
# Quick linear separability check: linear SVM on DINO features (LOO-CV)
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, LeaveOneOut

scaler = StandardScaler()
X = scaler.fit_transform(embeddings)
y = np.array(labels)

# Use stratified k-fold (5-fold) if enough samples, else LOO
n = len(X)
min_class_count = min(Counter(labels).values())
n_splits = min(5, min_class_count)

if n_splits >= 2:
    from sklearn.model_selection import StratifiedKFold
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(LinearSVC(max_iter=5000, C=1.0), X, y, cv=cv, scoring='accuracy')
    print(f'Linear SVM ({n_splits}-fold CV) on DINO features: {scores.mean():.1%} ± {scores.std():.1%}')
else:
    # LOO for very small datasets
    scores = cross_val_score(LinearSVC(max_iter=5000, C=1.0), X, y, cv=LeaveOneOut(), scoring='accuracy')
    print(f'Linear SVM (LOO) on DINO features: {scores.mean():.1%}')

# Chance level
print(f'Chance level: {1/len(set(labels)):.1%}')

## 5. Temporal Sequence: Attention Shift Across a Rally

Plot how the DINO attention _centre of mass_ moves over time during a rally. Does it track the players / shuttle?
This is a proxy for whether ViT provides complementary temporal signals.

In [ ]:
# Pick one rally and sample every 5th frame within the first shot's span
sample_rally = all_rallies[0]
vid = sample_rally['video'].replace('.mp4', '')
first_hit = sample_rally['hitting'][0]
start_f, end_f = first_hit['start_frame'], first_hit['end_frame']

frame_nums = list(range(start_f, end_f + 1, 3))  # every 3rd frame
print(f'Rally: {vid} | Shot frames {start_f}→{end_f} | Sampling {len(frame_nums)} frames')

att_centres_x, att_centres_y = [], []
for fn in frame_nums:
    try:
        img = load_frame(vid, fn)
        att = get_dino_attentions(img).mean(axis=0)  # (14, 14)
        att_norm = att / att.sum()
        ys, xs = np.meshgrid(np.linspace(0, 1, att.shape[0]),
                             np.linspace(0, 1, att.shape[1]), indexing='ij')
        cx = (att_norm * xs).sum()
        cy = (att_norm * ys).sum()
        att_centres_x.append(cx)
        att_centres_y.append(cy)
    except FileNotFoundError:
        att_centres_x.append(np.nan)
        att_centres_y.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
t = np.array(frame_nums) - start_f
axes[0].plot(t, att_centres_x, marker='o', ms=4, label='attention centre X (left→right)')
axes[0].axvline(first_hit['hit_frame'] - start_f, color='r', linestyle='--', label='hit frame')
axes[0].set_xlabel('Frame offset'); axes[0].set_ylabel('Normalised X'); axes[0].legend(); axes[0].set_title('Attention Centre X over time')

axes[1].plot(t, att_centres_y, marker='o', ms=4, color='orange', label='attention centre Y (top→bottom)')
axes[1].axvline(first_hit['hit_frame'] - start_f, color='r', linestyle='--', label='hit frame')
axes[1].set_xlabel('Frame offset'); axes[1].set_ylabel('Normalised Y'); axes[1].legend(); axes[1].set_title('Attention Centre Y over time')

plt.suptitle(f'DINO Attention Centre of Mass — {vid} shot 1', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Patch Relevance: Where Does ViT Focus vs. Skeleton Joints?

Overlay DINO attention on a frame and also draw the YOLOv8 skeleton joints from the `.npy` file. This tells us whether ViT and skeleton-based approaches attend to the same regions (redundant) or different ones (complementary).

In [ ]:
SKELETON_DIR = FB_SKELETONS_GDINO_V2

def load_skeleton(rally_id: str, frame_offset: int):
    """Load GDINO v2 skeleton npy. Shape (2, T, 34) → returns (34, 2) at frame_offset."""
    npy_path = SKELETON_DIR / f'{rally_id}.npy'
    if not npy_path.exists():
        return None
    sk = np.load(npy_path)  # (2, T, 34) — dim0=[x,y], dim1=frames, dim2=joints
    if frame_offset >= sk.shape[1]:
        return None
    # Reshape to (34, 2)
    return np.stack([sk[0, frame_offset], sk[1, frame_offset]], axis=-1)  # (34, 2)


# Show for one shot per strategy
fig, axes = plt.subplots(len(seen), 2, figsize=(12, 4*len(seen)))

for row, (strategy, s) in enumerate(seen.items()):
    img = load_frame(s['rally'], s['hit_frame'])
    att = get_dino_attentions(img)
    mean_att = att.mean(axis=0)
    att_norm = (mean_att - mean_att.min()) / (mean_att.max() - mean_att.min() + 1e-8)
    att_resized = np.array(Image.fromarray((att_norm * 255).astype(np.uint8)).resize(
        img.size, Image.BILINEAR)) / 255.0

    # Get skeleton: frame_offset = hit_frame - start_frame of the rally
    rally_meta = next(r for r in all_rallies if r['video'].replace('.mp4','') == s['rally'])
    frame_offset = s['hit_frame'] - rally_meta['start_frame']
    sk = load_skeleton(s['rally'], frame_offset)

    ax_att = axes[row, 0]
    ax_att.imshow(img)
    ax_att.imshow(att_resized, alpha=0.5, cmap='hot')
    ax_att.set_title(f"DINO attention\n{strategy} | {s['hit_type']}", fontsize=9)
    ax_att.axis('off')

    ax_sk = axes[row, 1]
    ax_sk.imshow(img)
    if sk is not None:
        # P0 = joints 0-16, P1 = joints 17-33
        for pidx, color in [(slice(0, 17), 'cyan'), (slice(17, 34), 'lime')]:
            pts = sk[pidx]  # (17, 2)
            valid = (pts[:, 0] > 0) & (pts[:, 1] > 0)
            ax_sk.scatter(pts[valid, 0], pts[valid, 1], c=color, s=15, zorder=5)
    ax_sk.set_title('Skeleton joints (GDINO v2)\nCyan=P0, Green=P1', fontsize=9)
    ax_sk.axis('off')

plt.suptitle('DINO Attention vs. Skeleton Joint Locations', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Summary: ViT Utility Assessment

| Use Case | Finding | Verdict |
|----------|---------|---------|
| **DINO attention maps** | Do heads highlight players/court/shuttle? | ✅ Free segmentation signal |
| **CLIP zero-shot** | Accuracy on raw hit frames? | 🔬 Experiment above |
| **CLS embedding clustering** | Linear SVM accuracy on DINO features? | 🔬 Experiment above |
| **Temporal attention tracking** | Does attention centre move with rally? | 🔬 Experiment above |
| **ViT vs skeleton overlap** | Do attention regions match joints? | 🔬 Visual inspection above |

---

### Concrete integration paths into the main pipeline:

**Path A — Feature Fusion (highest value, medium effort)**
- Extract DINO CLS token at hit frame → concat with ST-GCN embedding before prototypical head.
- ViT captures court context, player positioning, background court lines — things the skeleton misses.
- Expected gain: ~2-5% accuracy from richer context.

**Path B — DINO for Shuttle Detection (niche but elegant)**
- DINO attention heads sometimes attend to small moving objects (the shuttle is ~6×6px).
- If a specific head reliably fires on the shuttle, use that as a free shuttle detector.
- Alternative to running TrackNetV2 separately.

**Path C — CLIP for Annotation Bootstrap (low effort, useful)**
- Use CLIP zero-shot to pseudo-label unannotated ShuttleSet shots.
- Then use pseudo-labels as noisy weak supervision in SSL pre-training.
- No manual labelling needed.

**Path D — ViT Encoder as SSL Pre-training Target (high effort, research-grade)**
- Use frozen DINO features as the positive/negative signal for SimCLR instead of raw augmentation.
- Two views of the same skeleton that DINO says are similar → pull together.
- Could improve representation quality without more labels.

In [ ]:
# Optional: quick CLIP image embedding clustering (for comparison with DINO)
print('Extracting CLIP image embeddings for all shots...')
clip_embs = []
for s in shots:
    img = load_frame(s['rally'], s['hit_frame'])
    inputs = clip_processor(images=img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
    emb = F.normalize(emb, dim=-1).cpu().numpy()[0]
    clip_embs.append(emb)
clip_embs = np.stack(clip_embs)

# t-SNE comparison: DINO vs CLIP
tsne2 = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
clip_2d = tsne2.fit_transform(clip_embs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (data, title) in zip(axes, [(emb_2d, 'DINO ViT-B/16'), (clip_2d, 'CLIP ViT-B/32')]):
    for cls, col in STRATEGY_COLORS.items():
        idx = [i for i, l in enumerate(labels) if l == cls]
        if idx:
            ax.scatter(data[idx, 0], data[idx, 1], c=col, label=cls, s=60, alpha=0.8, edgecolors='k', linewidths=0.4)
    ax.set_title(f't-SNE — {title} embeddings', fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('DINO vs CLIP: strategy cluster quality from raw hit frames', fontsize=12)
plt.tight_layout()
plt.show()